# 09 — No-text ablation (the paper's headline contribution)

Identical pipeline to notebooks 06+07, but the ranker training **excludes
text features** (PCA dims + text_novelty + days_since_filing + doc_count_7d).
This isolates the *marginal value of text* in the full pipeline.

Outputs land in `artifacts/no-text/walk-{N:03d}/` (ranker + scoreboard) and
`artifacts/no-text/rl/walk-{N:03d}/cost-005bps/` (PPO).

Comparison vs full-text pipeline happens in notebook 08 (extension).

**Run order**: A (setup) → B (helpers) → C (walk-1 ranker, Optuna) →
D (walks 2-16 ranker, frozen HPs) → E (scoreboards 1-16) → F (PPO 1-16) → G (summary).

Resumable at every step. F is the long pole (~50 min/walk × 16 = 13h overnight).


## A. Setup

In [ ]:
from __future__ import annotations
import json, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import optuna
from lightgbm import early_stopping
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, ProgressBarCallback,
)

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import (
    assemble_walk_features,
    build_ranker,
    compute_excess_return_buckets,
    drop_zero_info_columns,
    evaluate_ranker,
    friday_only,
)
from src.utils.rl_env import (
    PortfolioEnv,
    build_scoreboard_from_scored_panel,
)

WALK_RANGE = range(1, 17)

PANEL_DIR = processed_dir() / 'training_panel'
OUT_ROOT  = repo_root() / 'artifacts' / 'no-text'
RL_OUT_ROOT = OUT_ROOT / 'rl'
OUT_ROOT.mkdir(parents=True, exist_ok=True)
RL_OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Ranker (notebook 06) parameters held constant for fair ablation.
N_BUCKETS = 32
TOP_K = 30
N_OPTUNA_TRIALS = 15

# PPO (notebook 07) parameters held constant for fair ablation.
COST_BPS = 5
EPISODE_LENGTH = 52
MAX_WEIGHT = 1.0
TOTAL_TIMESTEPS = 2_000_000
EVAL_FREQ = 10_000
CHECKPOINT_FREQ = 200_000
N_ENVS = 4
RANDOM_STATE = 42

# Text features to EXCLUDE for the no-text ranker. PCA cols are pca_*; aux text
# cols are explicit names.
TEXT_COLS_AUX = {'text_novelty', 'days_since_filing', 'doc_count_7d'}

print(f'no-text ablation: walks {min(WALK_RANGE)}..{max(WALK_RANGE)}; '
      f'{TOTAL_TIMESTEPS:,} PPO timesteps each')
print(f'output: {OUT_ROOT.relative_to(repo_root())}/, {RL_OUT_ROOT.relative_to(repo_root())}/')

## B. Helpers (no-text feature filter, ranker trainer, scoreboard builder)

In [ ]:
def _load_panel_window(start: str, end: str) -> pd.DataFrame:
    """Load training_panel rows in [start, end]."""
    s, e = pd.Timestamp(start), pd.Timestamp(end)
    frames = []
    for y in range(s.year, e.year + 1):
        for p in sorted((PANEL_DIR / f'year={y}').glob('*.parquet')):
            df = pd.read_parquet(p)
            df['date'] = pd.to_datetime(df['date'])
            df = df[(df['date'] >= s) & (df['date'] <= e)]
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def _walk_windows(walk_id: int) -> tuple[str, str, str, str, str, str]:
    """Walk N: train 2002-01-01..(2007+N-1)-12-31, val (2008+N-1), test (2009+N-1)."""
    train_end_year = 2007 + walk_id - 1
    return (
        '2002-01-01', f'{train_end_year}-12-31',
        f'{train_end_year + 1}-01-01', f'{train_end_year + 1}-12-31',
        f'{train_end_year + 2}-01-01', f'{train_end_year + 2}-12-31',
    )


def assemble_no_text_features(panel: pd.DataFrame, target_col: str = 'fwd_ret_5d'):
    """Same as src.utils.ranker.assemble_walk_features but with no PCA join and
    text-aux columns excluded. Returns (X, y_excess, group_sizes, meta)."""
    fri = friday_only(panel).dropna(subset=[target_col]).sort_values(['date', 'permno']).reset_index(drop=True)

    # Build an empty embed_pca-like frame so assemble_walk_features works uniformly.
    empty_pca = pd.DataFrame({'permno': fri['permno'], 'date': fri['date']})
    # Use assemble_walk_features for consistency with notebook 06's universe filter / drop logic,
    # then strip out aux text features afterward.
    X, y, groups, meta = assemble_walk_features(fri, empty_pca, target_col=target_col)
    # Drop aux text features if present (PCA cols won't be there since we passed empty_pca).
    drop_cols = [c for c in X.columns if c in TEXT_COLS_AUX]
    if drop_cols:
        X = X.drop(columns=drop_cols)
    return X, y, groups, meta


def train_no_text_ranker_for_walk(walk_id: int, hps: dict | None = None,
                                  run_optuna: bool = False) -> dict:
    """Fit a no-text LGBMRanker on walk-N's train+val windows.

    If `run_optuna=True`, runs 15-trial Optuna search on val NDCG@30 (use for
    walk-1 only). Otherwise uses provided `hps` (frozen from walk-1).
    """
    tr_s, tr_e, vl_s, vl_e, _, _ = _walk_windows(walk_id)
    panel_train = _load_panel_window(tr_s, tr_e)
    panel_val   = _load_panel_window(vl_s, vl_e)

    X_tr, y_tr, g_tr, m_tr = assemble_no_text_features(panel_train)
    X_vl, y_vl, g_vl, m_vl = assemble_no_text_features(panel_val)
    X_tr, X_vl = drop_zero_info_columns(X_tr, X_vl)

    b_tr = compute_excess_return_buckets(m_tr, n_buckets=N_BUCKETS).astype(int).values
    b_vl = compute_excess_return_buckets(m_vl, n_buckets=N_BUCKETS).astype(int).values

    if run_optuna:
        def _objective(trial: optuna.Trial) -> float:
            params = {
                'num_leaves':       trial.suggest_int('num_leaves', 15, 127),
                'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
                'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 20, 200),
                'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
                'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
                'lambda_l2':        trial.suggest_float('lambda_l2', 0.0, 5.0),
                'n_estimators':     500,
            }
            m = build_ranker(params)
            m.fit(X_tr, b_tr, group=g_tr,
                  eval_set=[(X_vl, b_vl)], eval_group=[g_vl], eval_at=[TOP_K],
                  callbacks=[early_stopping(stopping_rounds=30, verbose=False)])
            return float(m.best_score_['valid_0'][f'ndcg@{TOP_K}'])

        study = optuna.create_study(
            direction='maximize',
            pruner=optuna.pruners.MedianPruner(),
            sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        )
        study.optimize(_objective, n_trials=N_OPTUNA_TRIALS, show_progress_bar=True)
        best_hps = study.best_params
        print(f'  walk {walk_id} Optuna best NDCG@{TOP_K}: {study.best_value:.4f}')
    else:
        assert hps is not None, 'hps required when run_optuna=False'
        best_hps = hps

    final_hps = {**best_hps, 'n_estimators': 2000}
    model = build_ranker(final_hps)
    model.fit(X_tr, b_tr, group=g_tr,
              eval_set=[(X_vl, b_vl)], eval_group=[g_vl], eval_at=[TOP_K],
              callbacks=[early_stopping(stopping_rounds=50, verbose=False)])

    # Persist.
    walk_dir = OUT_ROOT / f'walk-{walk_id:03d}'
    walk_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump({'model': model, 'feature_names': X_tr.columns.tolist(),
                 'hps': best_hps}, walk_dir / 'ranker.joblib')

    return {
        'walk_id': walk_id,
        'best_hps': best_hps,
        'n_features': int(X_tr.shape[1]),
        'best_iteration': int(model.best_iteration_),
        f'val_ndcg_at_{TOP_K}': float(model.best_score_['valid_0'][f'ndcg@{TOP_K}']),
    }


def build_no_text_scoreboard_for_walk(walk_id: int) -> pd.DataFrame:
    """Score the panel (train+val+test windows) with walk-N's no-text ranker;
    return top-K per Friday scoreboard."""
    tr_s, tr_e, vl_s, vl_e, te_s, te_e = _walk_windows(walk_id)
    bundle = joblib.load(OUT_ROOT / f'walk-{walk_id:03d}' / 'ranker.joblib')
    model, features = bundle['model'], bundle['feature_names']

    panel = _load_panel_window(tr_s, te_e)
    fri = friday_only(panel).dropna(subset=['fwd_ret_5d']).copy()
    X = pd.DataFrame({c: fri[c] if c in fri.columns else np.nan for c in features})
    fri['score'] = model.predict(X)
    return build_scoreboard_from_scored_panel(fri, top_k=TOP_K)


print('helpers defined: _walk_windows, assemble_no_text_features, '
      'train_no_text_ranker_for_walk, build_no_text_scoreboard_for_walk')

## C. Walk-1 no-text ranker with 15-trial Optuna

In [ ]:
walk1_hps_path = OUT_ROOT / 'walk1_hps.json'

if walk1_hps_path.exists() and (OUT_ROOT / 'walk-001' / 'ranker.joblib').exists():
    walk1_info = json.loads(walk1_hps_path.read_text())
    print(f'walk-1 already fit; best HPs: {walk1_info["best_hps"]}')
else:
    print('fitting walk-1 no-text ranker with 15-trial Optuna...')
    walk1_info = train_no_text_ranker_for_walk(1, run_optuna=True)
    walk1_hps_path.write_text(json.dumps(walk1_info, indent=2))
    print(f'  saved -> {walk1_hps_path.relative_to(repo_root())}')

FROZEN_RANKER_HPS = walk1_info['best_hps']
print(f'frozen HPs for walks 2-16: {FROZEN_RANKER_HPS}')

## D. Walks 2-16 no-text ranker with frozen HPs

In [ ]:
ranker_summary = [walk1_info]
for w in range(2, 17):
    walk_dir = OUT_ROOT / f'walk-{w:03d}'
    if (walk_dir / 'ranker.joblib').exists():
        print(f'walk {w}: ranker exists — skipping')
        continue
    print(f'\nwalk {w}: fitting no-text ranker (frozen HPs)...')
    info = train_no_text_ranker_for_walk(w, hps=FROZEN_RANKER_HPS, run_optuna=False)
    ranker_summary.append(info)
    print(f'  val NDCG@{TOP_K}={info[f"val_ndcg_at_{TOP_K}"]:.4f}, '
          f'best_iter={info["best_iteration"]}, n_features={info["n_features"]}')

(OUT_ROOT / 'all_walks_ranker_summary.json').write_text(
    json.dumps({'walks': ranker_summary}, indent=2)
)
print(f'\nno-text rankers fit for {len(list(WALK_RANGE))} walks')

## E. Build no-text scoreboards (walks 1-16)

In [ ]:
for w in WALK_RANGE:
    sb_path = OUT_ROOT / f'walk-{w:03d}' / 'scoreboard.parquet'
    if sb_path.exists():
        print(f'walk {w}: scoreboard exists — skipping')
        continue
    print(f'walk {w}: building no-text scoreboard...')
    sb = build_no_text_scoreboard_for_walk(w)
    sb.to_parquet(sb_path, compression='zstd', index=False)
    print(f'  wrote {len(sb)} rows ({sb["date"].nunique()} Fridays)')

print(f'\nno-text scoreboards built for {len(list(WALK_RANGE))} walks')

## F. PPO training loop (walks 1-16, 5 bps)

In [ ]:
def _make_env_fn(scoreboard_subset: pd.DataFrame, seed: int):
    def _thunk():
        env = PortfolioEnv(
            scoreboard=scoreboard_subset, top_k=TOP_K,
            episode_length=EPISODE_LENGTH, cost_bps=float(COST_BPS),
            max_weight=MAX_WEIGHT,
        )
        env.reset(seed=seed)
        return Monitor(env)
    return _thunk


def train_ppo_for_walk(walk_id: int) -> dict:
    out_dir = RL_OUT_ROOT / f'walk-{walk_id:03d}' / f'cost-{COST_BPS:03d}bps'
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'ckpts').mkdir(exist_ok=True)
    (out_dir / 'tb').mkdir(exist_ok=True)
    if (out_dir / 'final_policy.zip').exists():
        existing = json.loads((out_dir / 'training_metrics.json').read_text())
        print(f'walk {walk_id}: PPO exists — skipping')
        return existing

    tr_s, tr_e, vl_s, vl_e, _, _ = _walk_windows(walk_id)
    sb = pd.read_parquet(OUT_ROOT / f'walk-{walk_id:03d}' / 'scoreboard.parquet')
    sb['date'] = pd.to_datetime(sb['date'])
    sb_tr = sb[(sb['date'] >= tr_s) & (sb['date'] <= tr_e)].copy()
    sb_vl = sb[(sb['date'] >= vl_s) & (sb['date'] <= vl_e)].copy()
    print(f'  walk {walk_id}: train Fridays={sb_tr["date"].nunique()}, '
          f'val Fridays={sb_vl["date"].nunique()}')

    train_vec = DummyVecEnv([_make_env_fn(sb_tr, RANDOM_STATE + i) for i in range(N_ENVS)])
    train_vec = VecNormalize(train_vec, norm_obs=True, norm_reward=False, clip_obs=10.0)
    val_vec = DummyVecEnv([_make_env_fn(sb_vl, RANDOM_STATE + 1000)])
    val_vec = VecNormalize(val_vec, norm_obs=True, norm_reward=False, clip_obs=10.0,
                           training=False)

    eval_cb = EvalCallback(val_vec, best_model_save_path=str(out_dir),
                           log_path=str(out_dir), eval_freq=max(EVAL_FREQ // N_ENVS, 1),
                           n_eval_episodes=1, deterministic=True)
    ckpt_cb = CheckpointCallback(save_freq=max(CHECKPOINT_FREQ // N_ENVS, 1),
                                 save_path=str(out_dir / 'ckpts'), name_prefix='ppo')

    model = PPO(
        'MlpPolicy', train_vec,
        policy_kwargs=dict(net_arch=[128, 64]),
        learning_rate=1e-4, n_steps=2048, batch_size=64, n_epochs=5,
        gamma=0.99, gae_lambda=0.95, clip_range=0.15,
        ent_coef=0.005, vf_coef=0.5, max_grad_norm=0.5, target_kl=0.03,
        device='cpu', tensorboard_log=str(out_dir / 'tb'),
        seed=RANDOM_STATE, verbose=0,
    )

    t0 = time.time()
    model.learn(total_timesteps=TOTAL_TIMESTEPS,
                callback=[eval_cb, ckpt_cb, ProgressBarCallback()], progress_bar=False)
    elapsed = time.time() - t0

    model.save(out_dir / 'final_policy')
    train_vec.save(str(out_dir / 'vec_normalize.pkl'))

    metrics = {
        'walk_id': walk_id, 'cost_bps': COST_BPS,
        'total_timesteps': TOTAL_TIMESTEPS, 'n_envs': N_ENVS,
        'wall_time_sec': elapsed, 'wall_time_hr': elapsed / 3600.0,
        'best_val_mean_reward': float(eval_cb.best_mean_reward),
    }
    (out_dir / 'training_metrics.json').write_text(json.dumps(metrics, indent=2))
    print(f'  walk {walk_id}: done; wall={elapsed/60:.1f} min; '
          f'best_val_mean_reward={metrics["best_val_mean_reward"]:.4f}')
    return metrics


ppo_summary = []
for w in WALK_RANGE:
    print(f'\n=== walk {w} ===')
    ppo_summary.append(train_ppo_for_walk(w))

(RL_OUT_ROOT / 'all_walks_summary.json').write_text(
    json.dumps({'walks': ppo_summary}, indent=2)
)
print(f'\nfinished no-text PPO for {len(ppo_summary)} walks')

## G. Summary

In [ ]:
# Quick at-a-glance comparison of no-text ranker quality across walks.
ranker_df = pd.DataFrame(ranker_summary).sort_values('walk_id').reset_index(drop=True)
print('=== no-text ranker per walk ===')
print(ranker_df[['walk_id', 'n_features', 'best_iteration', f'val_ndcg_at_{TOP_K}']]
      .to_string(index=False))

if ppo_summary:
    ppo_df = pd.DataFrame(ppo_summary).sort_values('walk_id').reset_index(drop=True)
    print('\n=== no-text PPO per walk ===')
    print(ppo_df[['walk_id', 'wall_time_hr', 'best_val_mean_reward']].to_string(index=False))

print(f'\nartifacts under: {OUT_ROOT.relative_to(repo_root())}/')
print('full vs no-text comparison happens in notebook 08 (extension).')